# SVAMITVA — Dual GPU Training
Uses both V100s via `torch.nn.DataParallel`.  
Automatically detects available GPUs and falls back to single GPU if needed.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — INSTALL
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "segmentation-models-pytorch", "albumentations",
     "rasterio", "scikit-learn", "timm", "geopandas",
     "shapely", "fiona", "scipy", "Pillow",
     "opencv-python-headless", "torchvision"],
    capture_output=True
)
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — GPU DETECTION
# Finds all free GPUs, picks the fastest one as primary,
# uses both if both are free.
# ─────────────────────────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess

def get_gpu_info():
    """Returns list of (gpu_id, free_mb, total_mb, name) sorted by free memory desc."""
    result = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=index,memory.free,memory.total,name",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    gpus = []
    for line in result.stdout.strip().split("\n"):
        if not line.strip():
            continue
        parts = [p.strip() for p in line.split(",")]
        gpus.append({
            "id":       int(parts[0]),
            "free_mb":  int(parts[1]),
            "total_mb": int(parts[2]),
            "name":     parts[3],
        })
    return sorted(gpus, key=lambda g: g["free_mb"], reverse=True)

gpu_info = get_gpu_info()
print("Available GPUs:")
for g in gpu_info:
    pct_free = 100 * g["free_mb"] / g["total_mb"]
    print(f"  GPU {g['id']}: {g['name']}  "
          f"{g['free_mb']/1024:.1f}GB free / {g['total_mb']/1024:.1f}GB total "
          f"({pct_free:.0f}% free)")

# Only use GPUs with >20GB free (enough for 1024px tiles)
FREE_THRESHOLD_MB = 20_000
usable = [g for g in gpu_info if g["free_mb"] > FREE_THRESHOLD_MB]

if len(usable) == 0:
    # Fall back to whatever is most free
    usable = [gpu_info[0]]
    print(f"\n⚠️  No GPU with >{FREE_THRESHOLD_MB//1024}GB free. Using GPU {usable[0]['id']} anyway.")

GPU_IDS  = [g["id"] for g in usable]
PRIMARY  = GPU_IDS[0]

# Tell PyTorch which GPUs to use
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in GPU_IDS)

print(f"\n{'─'*50}")
print(f"Using GPU(s)  : {GPU_IDS}")
print(f"Primary GPU   : {PRIMARY}")
print(f"Mode          : {'DataParallel (both GPUs)' if len(GPU_IDS) > 1 else 'Single GPU'}")
print(f"{'─'*50}")

Available GPUs:
  GPU 0: Tesla V100-PCIE-32GB  31.7GB free / 32.0GB total (99% free)
  GPU 1: Tesla V100-PCIE-32GB  31.7GB free / 32.0GB total (99% free)

──────────────────────────────────────────────────
Using GPU(s)  : [0, 1]
Primary GPU   : 0
Mode          : DataParallel (both GPUs)
──────────────────────────────────────────────────


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import gc, glob, json, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import rasterio
import rasterio.features
import rasterio.windows
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import shape
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib
matplotlib.use("Agg")   # no display needed on server
from tqdm import tqdm
from scipy.ndimage import binary_closing, binary_opening
import cv2
warnings.filterwarnings("ignore")

gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark    = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32   = True

# After setting CUDA_VISIBLE_DEVICES, device 0 = physical GPU[0] in our list
DEVICE = torch.device("cuda:0")

# Remapped device indices for DataParallel
# If we have 2 GPUs, device_ids=[0,1]; if 1 GPU, device_ids=[0]
n_gpus = torch.cuda.device_count()
DP_IDS = list(range(n_gpus))   # [0] or [0, 1]

print(f"torch.cuda.device_count() : {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    free = torch.cuda.mem_get_info(i)[0] / 1e9
    print(f"  cuda:{i}  {name}  {free:.1f}/{mem:.1f}GB free")

torch.cuda.device_count() : 2
  cuda:0  Tesla V100-PCIE-32GB  33.7/34.1GB free
  cuda:1  Tesla V100-PCIE-32GB  33.7/34.1GB free


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — PATHS + CLASS CONFIG
# ─────────────────────────────────────────────────────────────────────────────
BASE     = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"
CKPT_DIR = f"{BASE}/CHECKPOINTS_DUAL"
OUT_DIR  = f"{BASE}/FINAL_OUTPUTS"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

IMG_DIR  = f"{BASE}/trainingdata/DATASET_1024/images"
MASK_DIR = f"{BASE}/trainingdata/DATASET_1024/masks"

# Test villages — update path if yours is different
TEST_TIFS = sorted(glob.glob(f"{BASE}/live_demo/*.tif"))
if not TEST_TIFS:
    # Try common alternative locations
    for candidate in [
        f"{BASE}/PB_live_demo/*.tif",
        f"{BASE}/test/*.tif",
        f"{BASE}/*.tif",
    ]:
        TEST_TIFS = sorted(glob.glob(candidate))
        if TEST_TIFS:
            print(f"Found test TIFs at: {candidate}")
            break

print(f"Test villages : {len(TEST_TIFS)}")
for t in TEST_TIFS[:5]:
    print(f"  {os.path.basename(t)}")

# ── Class remapping ───────────────────────────────────────────────────────────
# Original 9-class masks → 4 clean classes
# 0=bg  1=built_up  2=road  3=road_cl  4=railway  5=bridge
# 6=water_line  7=waterbody_pt  8=utility  9=utility_poly
REMAP_ARRAY = np.array([
    0,  # 0  background        → background
    1,  # 1  built_up_area     → building
    2,  # 2  road              → road
    2,  # 3  road_centre_line  → road  (merge)
    0,  # 4  railway           → background (4 objects)
    0,  # 5  bridge            → background (8 objects)
    3,  # 6  water_body_line   → water
    3,  # 7  waterbody_point   → water
    0,  # 8  utility           → background
    0,  # 9  utility_poly      → background
], dtype=np.int64)

NUM_CLASSES    = 4
CLASS_NAMES    = ["background", "building", "road", "water"]
ACTIVE_CLASSES = [1, 2, 3]

COLOR_MAP = np.array([
    [0,   0,   0  ],   # background — black
    [255, 50,  50 ],   # building   — red
    [255, 200, 0  ],   # road       — yellow
    [0,   100, 255],   # water      — blue
], dtype=np.uint8)

PIXEL_COUNTS = np.array([
    1_764_710_366 + 543_668 + 93 + 666_574,
    388_862_220,
    49_832_874 + 131_981_531,
    44_439_682 + 53_439,
], dtype=np.float64)

print(f"\nClass distribution:")
total = PIXEL_COUNTS.sum()
for i, (n, c) in enumerate(zip(CLASS_NAMES, PIXEL_COUNTS)):
    print(f"  {i} {n:12s}: {int(c):>14,}  ({100*c/total:.1f}%)")

Found test TIFs at: /home/jupyter-228w1a1286/dinesh-data/hackthonndata/*.tif
Test villages : 5
  28996_NADALA_ORTHO.tif
  TIMMOWAL_37695_ORI.tif
  amora.tif
  bagga.tif
  fattu_bhila.tif

Class distribution:
  0 background  :  1,765,920,701  (74.2%)
  1 building    :    388,862,220  (16.3%)
  2 road        :    181,814,405  (7.6%)
  3 water       :     44,493,121  (1.9%)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — CONFIG
# Batch size is automatically scaled by number of GPUs.
# per_gpu_batch * n_gpus = effective batch per step.
# ─────────────────────────────────────────────────────────────────────────────
PER_GPU_BATCH = 4      # safe for 1024px on V100-32GB with grad checkpointing
EFFECTIVE_BATCH = PER_GPU_BATCH * n_gpus  # 8 with 2 GPUs, 4 with 1

CFG = {
    "img_dir":         IMG_DIR,
    "mask_dir":        MASK_DIR,
    "save_dir":        CKPT_DIR,

    "num_classes":     NUM_CLASSES,
    "tile_size":       768,          # 768 is the sweet spot: fits batch=4/GPU
                                     # change to 1024 only if you see <20GB usage

    "encoder":         "mit_b3",     # B3 with 2 GPUs = same VRAM per GPU as B2 solo
    "encoder_weights": "imagenet",

    "batch_size":      EFFECTIVE_BATCH,   # DataParallel splits this across GPUs
    "accum_steps":     2,                 # effective = EFFECTIVE_BATCH * 2
    "num_epochs":      60,
    "lr":              6e-5,
    "weight_decay":    1e-2,
    "val_split":       0.15,
    "num_workers":     4 * n_gpus,        # scale workers with GPUs

    "ce_weight":       0.35,
    "dice_weight":     0.45,
    "focal_weight":    0.20,

    "patience":        15,
    "min_delta":       5e-4,

    "resume":          False,
    "resume_ckpt":     f"{CKPT_DIR}/best_model.pth",
}

print(f"Per-GPU batch : {PER_GPU_BATCH}")
print(f"Effective batch (per accum step): {EFFECTIVE_BATCH}")
print(f"True effective batch: {EFFECTIVE_BATCH * CFG['accum_steps']}")
print(f"Encoder       : {CFG['encoder']}")
print(f"Tile size     : {CFG['tile_size']}px")
print(f"Num workers   : {CFG['num_workers']}")

Per-GPU batch : 4
Effective batch (per accum step): 8
True effective batch: 16
Encoder       : mit_b3
Tile size     : 768px
Num workers   : 8


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — DATASET + AUGMENTATIONS
# ─────────────────────────────────────────────────────────────────────────────
TILE = CFG["tile_size"]

class SVAMITVADataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32) / 255.0
            if img.shape[0] > 3:
                img = img[:3]
            img = np.transpose(img, (1, 2, 0))

        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)

        mask = REMAP_ARRAY[np.clip(mask, 0, len(REMAP_ARRAY) - 1)]

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug["image"]
            mask = aug["mask"].long()
        else:
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask).long()

        return img, mask

    def get_sample_weight(self, idx):
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = REMAP_ARRAY[np.clip(src.read(1).flatten(), 0, len(REMAP_ARRAY) - 1)]
        return float((mask > 0).sum()) + 1.0


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                       rotate_limit=20, p=0.5),
    A.RandomScale(scale_limit=0.3, p=0.4),
    A.PadIfNeeded(min_height=TILE, min_width=TILE,
                  border_mode=0, value=0, mask_value=0),
    A.RandomCrop(height=TILE, width=TILE),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.25),
        A.OpticalDistortion(distort_limit=0.2),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05),
    ], p=0.25),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.1),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8)),
    ], p=0.6),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 60)),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
    ], p=0.25),
    A.CoarseDropout(max_holes=8, max_height=48, max_width=48,
                    fill_value=0, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("Dataset and augmentations defined ✅")

Dataset and augmentations defined ✅


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
all_imgs  = sorted(glob.glob(f"{CFG['img_dir']}/*.tif"))
all_masks = sorted(glob.glob(f"{CFG['mask_dir']}/*.tif"))

assert len(all_imgs) > 0,              f"No tiles in {CFG['img_dir']}"
assert len(all_imgs) == len(all_masks), f"Mismatch: {len(all_imgs)} imgs vs {len(all_masks)} masks"

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(
    indices, test_size=CFG["val_split"], random_state=SEED
)

train_imgs  = [all_imgs[i]  for i in train_idx]
train_masks = [all_masks[i] for i in train_idx]
val_imgs    = [all_imgs[i]  for i in val_idx]
val_masks   = [all_masks[i] for i in val_idx]

for name, data in [("train_imgs",  train_imgs),  ("train_masks", train_masks),
                   ("val_imgs",    val_imgs),     ("val_masks",   val_masks)]:
    with open(os.path.join(CFG["save_dir"], f"{name}.json"), "w") as f:
        json.dump(data, f)

train_ds = SVAMITVADataset(train_imgs, train_masks, train_transform)
val_ds   = SVAMITVADataset(val_imgs,   val_masks,   val_transform)

# Weighted sampler — cache weights
cache = os.path.join(CFG["save_dir"], "sample_weights.json")
sw = None
if os.path.exists(cache):
    with open(cache) as f:
        sw = json.load(f)
    if len(sw) != len(train_ds):
        sw = None
    else:
        print("Loaded cached sample weights")

if sw is None:
    print("Computing sample weights (first run only)...")
    sw = [train_ds.get_sample_weight(i)
          for i in tqdm(range(len(train_ds)), desc="Weights")]
    with open(cache, "w") as f:
        json.dump(sw, f)

sampler = WeightedRandomSampler(
    weights=sw, num_samples=len(train_ds), replacement=True
)

# DataParallel requires shuffle=False when using a sampler
# pin_memory is important for fast CPU→GPU transfers
train_loader = DataLoader(
    train_ds,
    batch_size         = CFG["batch_size"],
    sampler            = sampler,
    num_workers        = CFG["num_workers"],
    pin_memory         = True,
    prefetch_factor    = 2,
    persistent_workers = True,
    drop_last          = True,
)
val_loader = DataLoader(
    val_ds,
    batch_size         = CFG["batch_size"],
    shuffle            = False,
    num_workers        = CFG["num_workers"],
    pin_memory         = True,
    prefetch_factor    = 2,
    persistent_workers = True,
)

print(f"Total tiles : {len(all_imgs)}")
print(f"Train       : {len(train_ds)}")
print(f"Val         : {len(val_ds)}")
print(f"Batch size  : {CFG['batch_size']} ({PER_GPU_BATCH}/GPU × {n_gpus} GPU)")

Loaded cached sample weights
Total tiles : 2744
Train       : 2332
Val         : 412
Batch size  : 8 (4/GPU × 2 GPU)


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — MODEL WITH DATAPARALLEL
# DataParallel splits each batch across GPUs, runs forward in parallel,
# gathers gradients on GPU 0. Simple and effective for 2 GPUs.
# ─────────────────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# Build base model on primary device
_base_model = smp.Segformer(
    encoder_name    = CFG["encoder"],
    encoder_weights = CFG["encoder_weights"],
    in_channels     = 3,
    classes         = CFG["num_classes"],
    activation      = None,
).to(DEVICE)

# Gradient checkpointing on encoder — critical for 768/1024px tiles
if hasattr(_base_model.encoder, "set_grad_checkpointing"):
    _base_model.encoder.set_grad_checkpointing(True)
    print("Gradient checkpointing: enabled")
else:
    for m in _base_model.encoder.modules():
        if hasattr(m, "gradient_checkpointing"):
            m.gradient_checkpointing = True
    print("Gradient checkpointing: enabled via module flag")

# Wrap with DataParallel if we have multiple GPUs
if len(DP_IDS) > 1:
    model = nn.DataParallel(_base_model, device_ids=DP_IDS)
    print(f"DataParallel: GPUs {DP_IDS}")
else:
    model = _base_model
    print(f"Single GPU mode: cuda:{DP_IDS[0]}")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model       : SegFormer-{CFG['encoder'].upper()} ({n_params:,} params)")

for i in range(n_gpus):
    free = torch.cuda.mem_get_info(i)[0] / 1e9
    print(f"VRAM free   : cuda:{i} → {free:.1f} GB")

# Helper to get base model regardless of DataParallel wrapping
def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

Gradient checkpointing: enabled via module flag
DataParallel: GPUs [0, 1]
Model       : SegFormer-MIT_B3 (44,598,980 params)
VRAM free   : cuda:0 → 33.5 GB
VRAM free   : cuda:1 → 33.7 GB


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — LOSS + OPTIMIZER + SCHEDULER
# ─────────────────────────────────────────────────────────────────────────────
# Class weights
raw_w  = 1.0 / np.log1p(PIXEL_COUNTS)
raw_w /= raw_w.mean()
raw_w  = np.clip(raw_w, 0, 15.0)
cw     = torch.tensor(raw_w, dtype=torch.float32).to(DEVICE)
print("Class weights:", [f"{w:.3f}" for w in raw_w])

# Losses (operate on the gathered output, so no special DP handling needed)
ce_loss    = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
dice_loss  = smp.losses.DiceLoss(
    mode="multiclass", classes=ACTIVE_CLASSES, smooth=1.0
)
focal_loss = smp.losses.FocalLoss(mode="multiclass", gamma=3.0)

def criterion(outputs, masks):
    return (CFG["ce_weight"]    * ce_loss(outputs, masks)    +
            CFG["dice_weight"]  * dice_loss(outputs, masks)  +
            CFG["focal_weight"] * focal_loss(outputs, masks))

# Separate LRs for encoder vs decoder
# Access base model params via unwrap()
enc_p, dec_p = [], []
for name, p in unwrap(model).named_parameters():
    (enc_p if "encoder" in name else dec_p).append(p)

optimizer = torch.optim.AdamW([
    {"params": enc_p, "lr": CFG["lr"],     "weight_decay": CFG["weight_decay"]},
    {"params": dec_p, "lr": CFG["lr"] * 5, "weight_decay": CFG["weight_decay"]},
])

def warmup_cosine(epoch, warmup=5, total=60):
    if epoch < warmup:
        return (epoch + 1) / warmup
    return 0.5 * (1.0 + np.cos(
        np.pi * (epoch - warmup) / (total - warmup)
    ))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda ep: warmup_cosine(ep, 5, CFG["num_epochs"]),
)

# GradScaler on primary device only (DataParallel handles the rest)
scaler = GradScaler("cuda")

print("Loss / optimizer / scheduler ready ✅")

Class weights: ['0.908', '0.978', '1.017', '1.098']
Loss / optimizer / scheduler ready ✅


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — METRICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(preds, masks):
    acc  = (preds == masks).sum().item() / masks.numel()
    ious = {}
    for cls in ACTIVE_CLASSES:
        inter = ((preds == cls) & (masks == cls)).sum().item()
        union = ((preds == cls) | (masks == cls)).sum().item()
        if union > 0:
            ious[cls] = inter / union
    miou = float(np.mean(list(ious.values()))) if ious else 0.0
    return acc, ious, miou

print("Metrics defined ✅")

Metrics defined ✅


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — RESUME (optional)
# ─────────────────────────────────────────────────────────────────────────────
best_acc    = 0.0
best_miou   = 0.0
start_epoch = 0

if CFG["resume"] and os.path.exists(CFG["resume_ckpt"]):
    print(f"Loading: {CFG['resume_ckpt']}")
    ckpt  = torch.load(CFG["resume_ckpt"], map_location=DEVICE, weights_only=False)
    state = {k.replace("_orig_mod.", ""): v
             for k, v in ckpt["model_state"].items()}
    unwrap(model).load_state_dict(state, strict=False)
    best_acc    = ckpt.get("val_acc",  0.0)
    best_miou   = ckpt.get("val_miou", 0.0)
    start_epoch = ckpt["epoch"]
    print(f"  Resumed epoch {start_epoch} | acc={best_acc*100:.2f}% | mIoU={best_miou:.4f}")
else:
    print("Training from scratch")

Training from scratch


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
history     = {"train_loss": [], "val_loss": [], "val_miou": [], "val_acc": []}
patience_ct = 0
nan_total   = 0

print("\n" + "═" * 72)
print(f"  SegFormer-{CFG['encoder'].upper()} | {n_gpus} GPU(s) | tile={CFG['tile_size']}px")
print(f"  Batch {PER_GPU_BATCH}/GPU × {n_gpus} GPU × accum {CFG['accum_steps']} "
      f"= {EFFECTIVE_BATCH * CFG['accum_steps']} true effective batch")
print(f"  Classes : {CLASS_NAMES}")
print(f"  Target  : ≥95% pixel accuracy")
print("═" * 72 + "\n")

for epoch in range(start_epoch, CFG["num_epochs"]):
    for i in range(n_gpus):
        torch.cuda.reset_peak_memory_stats(i)
    gc.collect()
    torch.cuda.empty_cache()
    t0 = time.time()

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss, valid_steps, nan_ep = 0.0, 0, 0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(
            train_loader, desc=f"Ep {epoch+1:03d} train", leave=False)):

        # DataParallel automatically splits batch across GPUs
        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            out  = model(imgs)
            loss = criterion(out, masks) / CFG["accum_steps"]

        if torch.isnan(loss) or torch.isinf(loss):
            nan_ep += 1
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            continue

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            has_nan = any(
                p.grad is not None and
                (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                for p in model.parameters()
            )
            if has_nan:
                nan_ep += 1
                optimizer.zero_grad()
                scaler.update()
                torch.cuda.empty_cache()
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss  += loss.item() * CFG["accum_steps"]
        valid_steps += 1

    scheduler.step()

    if nan_ep > 0:
        nan_total += nan_ep
        print(f"  ⚠️  {nan_ep} NaN batches (total: {nan_total})")
    if valid_steps == 0:
        print("  💀 All NaN — stopping.")
        break

    train_loss /= valid_steps
    train_t     = time.time() - t0

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_steps = 0.0, 0
    all_ious = {c: [] for c in ACTIVE_CLASSES}
    all_accs = []

    with torch.no_grad():
        for imgs, masks in tqdm(
                val_loader, desc=f"Ep {epoch+1:03d} val  ", leave=False):
            imgs  = imgs.to(DEVICE,  non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast("cuda"):
                out  = model(imgs)
                loss = criterion(out, masks)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            preds       = torch.argmax(out, dim=1)
            val_loss   += loss.item()
            val_steps  += 1
            acc, ious, _ = compute_metrics(preds, masks)
            all_accs.append(acc)
            for cls, iou in ious.items():
                all_ious[cls].append(iou)

    val_t         = time.time() - t0 - train_t
    val_loss     /= max(val_steps, 1)
    pc_iou        = {c: float(np.mean(v)) for c, v in all_ious.items() if v}
    val_miou      = float(np.mean(list(pc_iou.values()))) if pc_iou else 0.0
    val_acc       = float(np.mean(all_accs)) if all_accs else 0.0

    # Peak VRAM across all GPUs
    peak_mem = max(
        torch.cuda.max_memory_allocated(i) / 1e9
        for i in range(n_gpus)
    )

    torch.cuda.empty_cache()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["val_acc"].append(val_acc)

    improved = val_acc > best_acc + CFG["min_delta"]
    if improved:
        best_acc    = val_acc
        best_miou   = val_miou
        patience_ct = 0
        # Always save the unwrapped model state
        torch.save({
            "epoch":          epoch + 1,
            "model_state":    unwrap(model).state_dict(),
            "optim_state":    optimizer.state_dict(),
            "val_acc":        val_acc,
            "val_miou":       val_miou,
            "cfg":            CFG,
            "class_names":    CLASS_NAMES,
            "active_classes": ACTIVE_CLASSES,
            "remap_array":    REMAP_ARRAY.tolist(),
            "color_map":      COLOR_MAP.tolist(),
        }, os.path.join(CFG["save_dir"], "best_model.pth"))
        tag = f"  ✅ best  acc={best_acc*100:.2f}%  mIoU={best_miou:.4f}"
    else:
        patience_ct += 1
        tag = f"  (patience {patience_ct}/{CFG['patience']})"

    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"Ep {epoch+1:03d}/{CFG['num_epochs']} | "
        f"Train: {train_loss:.4f} ({train_t:.0f}s) | "
        f"Val: {val_loss:.4f} ({val_t:.0f}s) | "
        f"Acc: {val_acc*100:.2f}% | mIoU: {val_miou:.4f} | "
        f"VRAM: {peak_mem:.1f}GB | LR: {lr_now:.2e}{tag}"
    )

    if (epoch + 1) % 10 == 0:
        print("  Per-class IoU:")
        for cls in ACTIVE_CLASSES:
            print(f"    {cls} {CLASS_NAMES[cls]:10s}: {pc_iou.get(cls, 0.0):.4f}")

    if patience_ct >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch + 1}.")
        break

print(f"\nBest pixel accuracy : {best_acc*100:.2f}%")
print(f"Best mIoU           : {best_miou:.4f}")


════════════════════════════════════════════════════════════════════════
  SegFormer-MIT_B3 | 2 GPU(s) | tile=768px
  Batch 4/GPU × 2 GPU × accum 2 = 16 true effective batch
  Classes : ['background', 'building', 'road', 'water']
  Target  : ≥95% pixel accuracy
════════════════════════════════════════════════════════════════════════



Ep 001 train:   4%|▍         | 12/291 [01:01<15:18,  3.29s/it] 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train", lw=2)
axes[0].plot(ep, history["val_loss"],   label="Val",   lw=2)
axes[0].set_title("Loss", fontweight="bold")
axes[0].legend(); axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [v * 100 for v in history["val_acc"]],
             color="royalblue", lw=2, label="Pixel Acc %")
axes[1].axhline(95, color="red",    ls="--", lw=1.5, label="95% target")
axes[1].axhline(90, color="orange", ls="--", lw=1.5, label="90% target")
axes[1].set_title("Pixel Accuracy", fontweight="bold")
axes[1].legend(); axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history["val_miou"],
             color="seagreen", lw=2, label="mIoU")
axes[2].set_title("Val mIoU", fontweight="bold")
axes[2].legend(); axes[2].set_xlabel("Epoch"); axes[2].grid(alpha=0.3)

plt.suptitle(
    f"SegFormer-{CFG['encoder'].upper()} | {n_gpus} GPU(s) | tile={CFG['tile_size']}px",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
curve_path = os.path.join(CFG["save_dir"], "training_curves.png")
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved → {curve_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — LOAD BEST CHECKPOINT FOR INFERENCE
# Inference always runs on a single GPU (no DataParallel needed)
# ─────────────────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

INF_TILE    = 1024   # can be larger than training tile at inference
INF_OVERLAP = 256

inf_model = smp.Segformer(
    encoder_name    = CFG["encoder"],
    encoder_weights = None,
    in_channels     = 3,
    classes         = CFG["num_classes"],
).to(DEVICE)

ckpt_path = os.path.join(CFG["save_dir"], "best_model.pth")
ckpt      = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
state     = {k.replace("_orig_mod.", ""): v
             for k, v in ckpt["model_state"].items()}
inf_model.load_state_dict(state, strict=False)
inf_model.eval()

print(f"✅ Loaded checkpoint from epoch {ckpt['epoch']}")
print(f"   Val acc  : {ckpt['val_acc']*100:.2f}%")
print(f"   Val mIoU : {ckpt['val_miou']:.4f}")

inf_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — INFERENCE + VECTORIZATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def tta_predict(model, tensor):
    """Original + h-flip + v-flip, softmax-averaged."""
    with torch.no_grad():
        p0 = torch.softmax(model(tensor), dim=1)
        p1 = torch.flip(torch.softmax(model(torch.flip(tensor, [3])), dim=1), [3])
        p2 = torch.flip(torch.softmax(model(torch.flip(tensor, [2])), dim=1), [2])
    return ((p0 + p1 + p2) / 3.0)


def sliding_window_inference(tif_path):
    stride = INF_TILE - INF_OVERLAP
    with rasterio.open(tif_path) as src:
        H, W       = src.height, src.width
        profile    = src.profile.copy()
        transform_ = src.transform
        crs_       = src.crs
        output     = np.zeros((CFG["num_classes"], H, W), dtype=np.float32)
        count      = np.zeros((H, W), dtype=np.float32)

        for y in tqdm(range(0, H, stride),
                      desc=f"  {os.path.basename(tif_path)}"):
            for x in range(0, W, stride):
                wh = min(INF_TILE, H - y)
                ww = min(INF_TILE, W - x)
                img = src.read([1, 2, 3],
                               window=Window(x, y, ww, wh)
                               ).astype(np.float32) / 255.0
                img = np.transpose(img, (1, 2, 0))
                pad_h = INF_TILE - img.shape[0]
                pad_w = INF_TILE - img.shape[1]
                if pad_h > 0 or pad_w > 0:
                    img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)))
                t = inf_transform(image=img)["image"].unsqueeze(0).to(DEVICE)
                pred = tta_predict(inf_model, t).cpu().numpy()[0]
                pred = pred[:, :wh, :ww]
                output[:, y:y+wh, x:x+ww] += pred
                count[y:y+wh, x:x+ww]     += 1

    output /= np.maximum(count, 1)
    return np.argmax(output, axis=0).astype(np.uint8), profile, crs_, transform_


def heuristic_roof_type(chip_rgb):
    if chip_rgb.size == 0:
        return "Others"
    r, g, b = (chip_rgb[:, :, i].mean() for i in range(3))
    br = (r + g + b) / 3.0
    if br > 210:                              return "Tin"
    if r > g + 25 and r > b + 25:            return "Tiled"
    if abs(r-g) < 18 and abs(g-b) < 18 and br > 80: return "RCC"
    return "Others"


def prediction_to_gpkg(prediction, src_transform, src_crs,
                        ortho_path, output_gpkg, village_name):
    layer_cfg = {
        "buildings":    {"cls": 1, "min_m2": 15, "close": 3, "open": 2},
        "roads":        {"cls": 2, "min_m2": 5,  "close": 2, "open": 1},
        "water_bodies": {"cls": 3, "min_m2": 10, "close": 3, "open": 2},
    }
    px_area = abs(src_transform.a) * abs(src_transform.e)
    all_gdfs = {}

    with rasterio.open(ortho_path) as ortho:
        for lname, lc in layer_cfg.items():
            binary = (prediction == lc["cls"]).astype(np.uint8)
            binary = binary_closing(binary, iterations=lc["close"]).astype(np.uint8)
            binary = binary_opening(binary, iterations=lc["open"]).astype(np.uint8)

            min_px  = lc["min_m2"] / max(px_area, 1e-9)
            records = []
            for geom, val in rasterio.features.shapes(
                    binary, mask=binary, transform=src_transform):
                if val != 1: continue
                poly = shape(geom)
                if poly.area < min_px: continue
                rec = {
                    "geometry":     poly.simplify(0.5, preserve_topology=True),
                    "village":      village_name,
                    "feature_type": lname,
                    "area_m2":      round(poly.area * px_area, 2),
                    "roof_type":    None,
                }
                if lname == "buildings":
                    try:
                        win = rasterio.windows.from_bounds(
                            *poly.bounds, transform=ortho.transform
                        )
                        chip = np.transpose(
                            ortho.read([1, 2, 3], window=win), (1, 2, 0)
                        ).astype(np.uint8)
                        rec["roof_type"] = heuristic_roof_type(chip)
                    except Exception:
                        rec["roof_type"] = "Others"
                records.append(rec)

            if records:
                gdf = gpd.GeoDataFrame(records, crs=src_crs)
                all_gdfs[lname] = gdf
                print(f"  {lname:15s}: {len(gdf):5d} features")
            else:
                print(f"  {lname:15s}: 0 features")

    if os.path.exists(output_gpkg):
        os.remove(output_gpkg)
    for lname, gdf in all_gdfs.items():
        gdf.to_file(output_gpkg, layer=lname, driver="GPKG")
    print(f"  ✅ GeoPackage: {output_gpkg}")
    return all_gdfs


def save_cog(prediction, profile, out_path):
    profile.update(driver="GTiff", dtype=rasterio.uint8,
                   count=1, compress="lzw")
    tmp = out_path.replace(".tif", "_tmp.tif")
    with rasterio.open(tmp, "w", **profile) as dst:
        dst.write(prediction, 1)
    res = subprocess.run(
        ["gdal_translate", tmp, out_path,
         "-of", "COG", "-co", "COMPRESS=LZW",
         "-co", "OVERVIEWS=AUTO", "-co", "BLOCKSIZE=512"],
        capture_output=True, text=True
    )
    if res.returncode == 0:
        os.remove(tmp)
        print(f"  ✅ COG: {out_path}")
    else:
        os.rename(tmp, out_path)
        print(f"  ⚠️  Saved as regular GeoTIFF (gdal_translate not available)")

    # Color visualization
    col_path = out_path.replace(".tif", "_color.tif")
    col_profile = profile.copy()
    col_profile.update(count=3)
    color = COLOR_MAP[prediction]
    with rasterio.open(col_path, "w", **col_profile) as dst:
        dst.write(np.transpose(color, (2, 0, 1)))
    print(f"  ✅ Color viz: {col_path}")


print("Inference + vectorization helpers defined ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — RUN INFERENCE ON ALL TEST VILLAGES
# ─────────────────────────────────────────────────────────────────────────────
if not TEST_TIFS:
    print("⚠️  No test TIFs found. Update TEST_TIFS path in Cell 4.")
else:
    print(f"Running inference on {len(TEST_TIFS)} villages...\n")

    for tif_path in TEST_TIFS:
        village = os.path.splitext(os.path.basename(tif_path))[0]
        v_dir   = os.path.join(OUT_DIR, village)
        os.makedirs(v_dir, exist_ok=True)

        print(f"\n{'─'*60}")
        print(f"Village : {village}")

        pred, profile, crs_, tr_ = sliding_window_inference(tif_path)

        cog_path  = os.path.join(v_dir, f"{village}_prediction.tif")
        gpkg_path = os.path.join(v_dir, f"{village}_features.gpkg")

        save_cog(pred, profile, cog_path)
        gdfs = prediction_to_gpkg(pred, tr_, crs_, tif_path, gpkg_path, village)

        # Visualization
        with rasterio.open(tif_path) as src:
            orig = src.read([1, 2, 3])
        scale    = min(1.0, 2000 / max(orig.shape[1], orig.shape[2]))
        h_v      = int(orig.shape[1] * scale)
        w_v      = int(orig.shape[2] * scale)
        orig_v   = cv2.resize(np.transpose(orig,   (1, 2, 0)), (w_v, h_v))
        pred_v   = cv2.resize(pred, (w_v, h_v), interpolation=cv2.INTER_NEAREST)
        color_v  = COLOR_MAP[pred_v]
        overlay  = orig_v.copy()
        overlay[pred_v > 0] = color_v[pred_v > 0]
        blended  = cv2.addWeighted(orig_v, 0.55, overlay, 0.45, 0)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        for ax, im, title in zip(
            axes,
            [orig_v, color_v, blended],
            ["Original", "Prediction", "Overlay"]
        ):
            ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis("off")

        legend = [Patch(facecolor=COLOR_MAP[i]/255, label=CLASS_NAMES[i])
                  for i in range(1, NUM_CLASSES)]
        axes[2].legend(handles=legend, loc="lower right", fontsize=9)
        plt.suptitle(f"SVAMITVA — {village}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        viz_path = os.path.join(v_dir, f"{village}_viz.png")
        plt.savefig(viz_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  ✅ Visualization: {viz_path}")

    print(f"\n{'═'*60}")
    print(f"✅ All villages done. Outputs in: {OUT_DIR}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 — FINAL REPORT
# ─────────────────────────────────────────────────────────────────────────────
from datetime import datetime

report = {
    "project":   "SVAMITVA AI Feature Extraction",
    "hackathon": "IIT Tirupati Geospatial Intelligence Hackathon 2025-26",
    "problem":   "Problem Statement 1",
    "generated": datetime.now().isoformat(),
    "hardware": {
        "gpus_used":    GPU_IDS,
        "n_gpus":       n_gpus,
        "parallel_mode": "DataParallel" if n_gpus > 1 else "Single GPU",
    },
    "model": {
        "architecture": f"SegFormer-{CFG['encoder'].upper()}",
        "tile_size":    CFG["tile_size"],
        "num_classes":  NUM_CLASSES,
        "class_names":  CLASS_NAMES,
    },
    "training": {
        "tiles_total":   len(all_imgs),
        "tiles_train":   len(train_ds),
        "tiles_val":     len(val_ds),
        "best_val_acc":  f"{best_acc*100:.2f}%",
        "best_val_miou": f"{best_miou:.4f}",
        "epochs_run":    len(history["train_loss"]),
    },
    "outputs": {
        "raster": "Cloud Optimized GeoTIFF (COG)",
        "vector": "GeoPackage (GPKG)",
        "layers": [
            "buildings (roof_type: RCC / Tiled / Tin / Others)",
            "roads",
            "water_bodies",
        ],
    },
}

report_path = os.path.join(OUT_DIR, "report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("📄 FINAL REPORT")
print("═" * 50)
print(f"  GPUs          : {GPU_IDS} ({n_gpus} × V100-32GB)")
print(f"  Model         : SegFormer-{CFG['encoder'].upper()}")
print(f"  Tile size     : {CFG['tile_size']}px")
print(f"  Val accuracy  : {best_acc*100:.2f}%")
print(f"  Val mIoU      : {best_miou:.4f}")
print(f"  Raster output : COG GeoTIFF")
print(f"  Vector output : GeoPackage")
print(f"  Vector layers : buildings / roads / water_bodies")
print(f"  Roof types    : RCC / Tiled / Tin / Others")
print(f"  Report        : {report_path}")